In [1]:
import pandas as pd
from pyscenic.aucell import aucell
from pyscenic.utils import load_motifs
from pyscenic.utils import GeneSignature
import numpy as np
import os
import pickle
import scanpy as sc

/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/ctxcore/__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
os.getcwd()

'/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend'

In [3]:
data_dir = "data"

In [4]:
# Read matrix from mtx file
matrix = sc.read_mtx("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_sparse.mtx").T
# Read genes and barcodes
genes = pd.read_csv("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_genes.tsv", sep="\t", header=None)
barcodes = pd.read_csv("/home/annikafomm/University_Subject_Repos/mopitas-mapra/backend/data/CID3586/count_matrix_barcodes.tsv", sep="\t", header=None)

,0
10,SAMD11
11,NOC2L
12,KLHL17
13,PLEKHN1
14,PERM1
15,RP11-54O7.17
16,HES4
17,ISG15
18,AGRN
19,RP11-54O7.18


In [5]:
anndata = sc.AnnData(X=matrix.X, obs=barcodes, var=genes)

/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/annikafomm/miniconda3/envs/aucell_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [13]:
# Normalize the data
sc.pp.normalize_total(anndata, target_sum=1e6)
# Logarithmize the data
sc.pp.log1p(anndata)


In [14]:
# Filter
sc.pp.filter_cells(anndata, min_genes=200)
sc.pp.filter_genes(anndata, min_cells=3)

In [6]:
anndata.shape

(6178, 29733)

In [28]:
# convert to df
df = pd.DataFrame(anndata.X.toarray(), index=anndata.obs.index, columns=anndata.var[0])

In [16]:
# Read genie3 regulons
genie3 = pd.read_csv(
   f"{data_dir}/genie3_BRCA_mrn.top_100k.csv",
    index_col=0,
)

In [17]:
# Create GeneSets from genie3 regulons
genesets = []
for regulator, group in genie3.groupby("regulatoryGene"):
    gene2weight = dict(zip(group["targetGene"], group["weight"]))
    geneset = GeneSignature(name=regulator, gene2weight=gene2weight)
    genesets.append(geneset)

In [27]:
anndata.var

,0,n_cells
2,FO538757.2,442
3,AP006222.2,46
5,RP5-857K21.4,3
6,RP11-206L10.9,104
7,LINC00115,61
...,...,...
21159,AL354822.1,9
21160,AC004556.1,55
21161,AC233755.2,6
21162,AC233755.1,6


In [29]:
# Overlap regulons with genes in the data
genie3_genes = set(genie3["targetGene"].unique())
genes_in_data = set(df.columns)
overlap_genes = genie3_genes.intersection(genes_in_data)
print(f"Number of genes in genie3 regulons: {len(genie3_genes)}")
print(f"Number of genes in data: {len(genes_in_data)}")
print(f"Number of overlapping genes: {len(overlap_genes)}")


Number of genes in genie3 regulons: 17339
Number of genes in data: 17944
Number of overlapping genes: 14159


In [18]:
genesets

[GeneSignature(name='ADNP', gene2weight=frozendict.frozendict({'TOP2B': 0.0279155260750008, 'EGFL7': 0.0237409127547856, 'RASIP1': 0.0206077146851765, 'ABCD4': 0.0198264699957939, 'CLDN5': 0.0196923416113006, 'DCLRE1A': 0.0196791886852008, 'UBXN2B': 0.0186096779888699, 'DHX40': 0.0175653551076977, 'FEM1B': 0.0169275218517057, 'UPP1': 0.0167412548801931, 'INHBC': 0.0164379980511868, 'ZNF146': 0.0162366041101673, 'CLDN15': 0.0162188588231912, 'SUV420H1': 0.0162154303199169, 'FBXO21': 0.0161746744825105, 'LYL1': 0.0161460409470039, 'GTF3C4': 0.0159948001546787, 'CHML': 0.0155638331522917, 'PPP2R5B': 0.0153062509862445, 'EXOC3L2': 0.015090823235948, 'FES': 0.0149996808025288, 'RBAK': 0.014902061047451, 'KAT7': 0.0148831899298347, 'RASSF1': 0.014873103380482, 'TADA1': 0.0147882868342616, 'RBBP4': 0.0146250663555892, 'PLD3': 0.0145239365173512, 'TMEM150A': 0.014377200370458, 'IKBKG': 0.0143513116808776, 'SIN3A': 0.013912341210591, 'TNFRSF1A': 0.0138251453401087, 'SLC27A3': 0.0137148123313428

In [30]:
aucell_results = aucell(
    df,
    genesets,
    num_workers=8
)

Less than 80% of the genes in ZNF366 are present in the expression matrix.
Less than 80% of the genes in ETV5 are present in the expression matrix.
Less than 80% of the genes in ONECUT2 are present in the expression matrix.
Less than 80% of the genes in FAM200B are present in the expression matrix.
Less than 80% of the genes in AKAP8 are present in the expression matrix.
Less than 80% of the genes in OTX1 are present in the expression matrix.
Less than 80% of the genes in ALX4 are present in the expression matrix.
Less than 80% of the genes in ANKZF1 are present in the expression matrix.
Less than 80% of the genes in ARHGAP35 are present in the expression matrix.
Less than 80% of the genes in JRK are present in the expression matrix.
Less than 80% of the genes in ARID3A are present in the expression matrix.
Less than 80% of the genes in TBPL1 are present in the expression matrix.
Less than 80% of the genes in TBX1 are present in the expression matrix.
Less than 80% of the genes in PAX2

In [35]:
# How many zero
zero_aucell = np.sum(aucell_results > 0.4)
print(f"Number of zero AUC values: {zero_aucell}")

Number of zero AUC values: Regulon
ADNP      0
ADNP2     2
AEBP1     0
AEBP2     0
AHCTF1    0
         ..
ZUFSP     1
ZXDA      0
ZXDB      0
ZXDC      0
ZZZ3      0
Length: 1317, dtype: int64


/home/annikafomm/.local/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [32]:
aucell_results

Regulon,ADNP,ADNP2,AEBP1,AEBP2,AHCTF1,AHDC1,AHR,AKAP8,AKAP8L,AKNA,...,ZSCAN30,ZSCAN31,ZSCAN32,ZSCAN5A,ZSCAN9,ZUFSP,ZXDA,ZXDB,ZXDC,ZZZ3
Cell,,,,,,,,,,,,,,,,,,,,,
0,0.073757,0.000000,0.026134,0.012075,0.037737,0.021775,0.025082,0.0,0.024222,0.000000,...,0.0,0.093116,0.0,0.137058,0.0,0.000000,0.070847,0.0,0.063067,0.015920
1,0.061586,0.104632,0.014302,0.043846,0.044343,0.007694,0.043143,0.0,0.000456,0.017544,...,0.0,0.000000,0.0,0.076432,0.0,0.102862,0.059253,0.0,0.037151,0.036915
2,0.106515,0.000000,0.042316,0.021742,0.021213,0.019465,0.013981,0.0,0.012200,0.000000,...,0.0,0.000000,0.0,0.000000,0.0,0.272715,0.087389,0.0,0.053213,0.020991
3,0.074674,0.000000,0.027540,0.022461,0.040027,0.011126,0.049362,0.0,0.022232,0.000000,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.070933,0.0,0.074991,0.036627
4,0.093071,0.000000,0.039556,0.023466,0.021460,0.008754,0.024180,0.0,0.036773,0.054009,...,0.0,0.000000,0.0,0.071885,0.0,0.000000,0.000000,0.0,0.035081,0.050394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6173,0.016139,0.006649,0.004530,0.020003,0.041510,0.028890,0.014931,0.0,0.019643,0.025924,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.033083,0.0,0.057058,0.019234
6174,0.002398,0.000000,0.000000,0.013655,0.036230,0.003913,0.007890,0.0,0.000000,0.027476,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.073057,0.0,0.077264,0.018470
6175,0.005618,0.000000,0.003601,0.055859,0.018742,0.013662,0.040363,0.0,0.028865,0.008627,...,0.0,0.111649,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.071542,0.018189
